In [1]:
import requests
import pandas as pd 

### Punto 1: Selección del dataset
### Nombre del dataset: Crecimiento del Parque Automotor RUNT 2.0
### Fuente: Portal de Datos Abiertos de Colombia (datos.gov.co)
### Entidad: Ministerio de Transporte - MinTransporte, Bogotá D.C.
### URL: https://www.datos.gov.co/Transporte/CRECIMIENTO-DEL-PARQUE-AUTOMOTOR-RUNT2-0/u3vn-bdcy/about_data
### Endpoint API: https://www.datos.gov.co/api/v3/views/u3vn-bdcy/query.json
### ID del dataset: u3vn-bdcy
### Fecha de consulta: 15 de octubre de 2026
### Corte de los datos: octubre de 2026 (según mes_de_publicacion y a_o_de_publicacion)

### Punto 2

In [2]:
url = "https://www.datos.gov.co/api/v3/views/u3vn-bdcy/query.json"
response = requests.post(url = url)

In [3]:
response.status_code

200

In [4]:
df =pd.DataFrame(response.json())
df


,:id,:version,:created_at,:updated_at,nombre_departamento,nombre_municipio,nombre_servicio,estado_del_vehiculo,nombre_de_la_clase,fecha_de_registro,cantidad,mes_de_publicacion,a_o_de_publicacion
0,row-sad7-zayi~dwvn,rv-yi7p_bc2j_z26q,2026-10-05T17:20:11.664Z,2026-10-05T17:20:11.664Z,ANTIOQUIA,ANDES,PARTICULAR,ACTIVO,CAMPERO,1900,1,OCTUBRE,2026
1,row-9fhz~3i2a~duh3,rv-ymi4~bvfk.tzgu,2026-10-05T17:20:11.664Z,2026-10-05T17:20:11.664Z,ATLANTICO,BARANOA,PARTICULAR,ACTIVO,AUTOMOVIL,1900,1,OCTUBRE,2026
2,row-8eei-grs6_z8qe,rv-36fg-h8fi.ecn7,2026-10-05T17:20:11.664Z,2026-10-05T17:20:11.664Z,ATLANTICO,BARANOA,PÚBLICO,ACTIVO,CAMION,1900,1,OCTUBRE,2026
3,row-irb4_e4ca.dpie,rv-mr98-avgd.vqck,2026-10-05T17:20:11.664Z,2026-10-05T17:20:11.664Z,BOGOTA D.C.,BOGOTA,OFICIAL,ACTIVO,AUTOMOVIL,1900,1,OCTUBRE,2026
4,row-3r4y-w3sm-ydxy,rv-7nas-s3qy_fga5,2026-10-05T17:20:11.664Z,2026-10-05T17:20:11.664Z,BOGOTA D.C.,BOGOTA,PARTICULAR,ACTIVO,AUTOMOVIL,1900,1,OCTUBRE,2026
...,...,...,...,...,...,...,...,...,...,...,...,...,...
191917,row-j9ep~c3mt_qijw,rv-92y9~ftpy-hfq8,2026-10-05T17:20:11.664Z,2026-10-05T17:20:11.664Z,VALLE DEL CAUCA,ZARZAL,NO REGISTRA,ACTIVO,TRACTOR,2026,2,OCTUBRE,2026
191918,row-v8yc_vvpu_542t,rv-3rhy~e94h~ksfe,2026-10-05T17:20:11.664Z,2026-10-05T17:20:11.664Z,VICHADA,PUERTO CARRENO,OFICIAL,ACTIVO,CAMIONETA,2026,1,OCTUBRE,2026
191919,row-5igm.4huv.wnst,rv-c39a~ka2t_qsi5,2026-10-05T17:20:11.664Z,2026-10-05T17:20:11.664Z,VICHADA,PUERTO CARRENO,PARTICULAR,ACTIVO,MOTOCARRO,2026,206,OCTUBRE,2026
191920,row-dfcf~wtes~d8yk,rv-wwtd.5v4e~g7b5,2026-10-05T17:20:11.664Z,2026-10-05T17:20:11.664Z,VICHADA,PUERTO CARRENO,OFICIAL,ACTIVO,MOTOCARRO,2026,5,OCTUBRE,2026


In [5]:
df = df.drop(columns=[c for c in df.columns if c.startswith(":")])
df.columns

Index(['nombre_departamento', 'nombre_municipio', 'nombre_servicio',
       'estado_del_vehiculo', 'nombre_de_la_clase', 'fecha_de_registro',
       'cantidad', 'mes_de_publicacion', 'a_o_de_publicacion'],
      dtype='str')

Eliminamos las columnas que inician en :, porque no corresponden a ninguna variable de los datos

In [11]:
df.shape

(191922, 9)

El dataset tiene 191.922 filas y 9 columnas, despues de eliminar las columnas  que no tienen nada que ver con los datos.

In [6]:
df.duplicated().sum()

np.int64(0)

In [7]:
df.isnull().sum()

nombre_departamento    0
nombre_municipio       0
nombre_servicio        0
estado_del_vehiculo    0
nombre_de_la_clase     0
fecha_de_registro      0
cantidad               0
mes_de_publicacion     0
a_o_de_publicacion     0
dtype: int64

In [8]:
df.dtypes

nombre_departamento    str
nombre_municipio       str
nombre_servicio        str
estado_del_vehiculo    str
nombre_de_la_clase     str
fecha_de_registro      str
cantidad               str
mes_de_publicacion     str
a_o_de_publicacion     str
dtype: object

Observamos que el dataframe no presenta ningun valor nulo o duplicado, y que todas las variables vienen en formato de texto, asi que antes de empezar cualquier analisis se debe modificar el tipo de las variables numericas para que correspondan con su tipo.

In [9]:
df["cantidad"] = pd.to_numeric(df["cantidad"])
df["cantidad"].sum()

np.int64(22655040)

En total el dataset suma 22.655.040 vehiculos, una cifra cercana a los aprox 21 millones de vehiculos matriculados en Colombia con corte a 2025 segun la pagina del RUNT.

### Punto 3
# la unidad de analisis es: grupos de vehiculos registrados en el Runt 2.0

### cada fila no es un vehiculo, sino un grupo de vehiculos que comparten departamento, municipio, servicio, estado, clase y año de registro. por eso existe la columna cantidad, que dice cuantos vehiculos hay en cada grupo.

### en el dataset encontramos las siguientes variables para estudiar:
### nombre del departamento = departamento donde fue registrado 
### nombre del municipio = municipio desde donde fue registrado 
### nombre servicio = el servicio que proporciona ya sea particular, publico, oficial, etc.
### estado del vehiculo = si el vehiculo esta activo o no en el registro
### nombre de la clase = que tipo de vehiculo es (automovil, motocicleta, camion, etc.)
### fecha registro = año en que se registraron los vehiculos (es solo el año, no una fecha completa)
### cantidad = cantidad de vehiculos que hay en cada grupo
### mes publicacion = mes donde se publican los datos
### año de publicacion = año donde se publican los datos

### se eliminaron las columnas :id, :version, :created_at y :updated_at porque no son variables del dataset.

# Punto 4: Pregunta analítica

### ¿Cómo ha cambiado la cantidad de motocicletas registradas cada año en Colombia entre 2015 y 2025, en comparación con los automóviles?

### esta pregunta es relevante porque permite entender cómo han cambiado las preferencias de movilidad de los ciudadanos. la moto se ha convertido en una alternativa cada vez más usada frente al carro y al transporte público, posiblemente por ser más económica y rápida, y estos datos podrían dar indicios de qué tanto ha crecido esa preferencia.
### esto es importante porque la moto es uno de los medios de transporte con mas riesgos, y los motociclistas están involucrados en una gran parte de los siniestros viales del país. si su uso sigue en aumento, esta información puede servir para orientar campañas de prevención y formación sobre su uso correcto, y para planear proyectos de infraestructura vial que tengan en cuenta que son uno de los vehículos con mayor presencia en las vías.

### variables clave:
### fecha_de_registro = permite ver la evolución año por año
### nombre_de_la_clase = permite separar motocicletas y automóviles
### cantidad = permite sumar cuántos vehículos se registraron en cada año y clase

In [10]:
df["nombre_de_la_clase"].unique()

<StringArray>
[                                    'CAMPERO',
                                   'AUTOMOVIL',
                                      'CAMION',
                                 'MOTOCICLETA',
                                      'BUSETA',
                                   'CAMIONETA',
                                    'VOLQUETA',
                                    'MICROBUS',
                                'TRACTOCAMION',
                                         'BUS',
                                'SEMIREMOLQUE',
                                   'MOTOCARRO',
                             'MAQ. INDUSTRIAL',
                               'MAQ. AGRICOLA',
                                'MOTOTRICICLO',
                                   'SIN CLASE',
                                    'REMOLQUE',
                                  'CUATRIMOTO',
                                  'CICLOMOTOR',
                                 'MONTACARGAS',
                          


### Punto 5 limitaciones y riesgos
### 1. consistencia en las categorías: en nombre_de_la_clase los vehículos estan divididos en muchos tipos, lo cual complica responder la pregunta. para los carros habría que decidir si se cuentan solo los automóviles o tambien las camionetas y camperos, y con las motos pasa algo parecido ya que existen clases como motocarro, mototriciclo, cuatrimoto y ciclomotor. ademas aparece una categoría "SIN CLASE" y maquinaria como excavadoras mezclada con los demas vehículos. como el dataset no indica como agruparlos, el resultado puede cambiar según la decisión que se tome.

### 2. nivel de agregación: como cada fila es un grupo de vehículos y no una persona, el dataset no tiene informacion de quienes compran las motos ni que medio de transporte usaban antes. por eso aunque los datos pueden mostrar si las motos crecen mas que los carros, no permiten confirmar que las personas esten cambiando el carro o el transporte publico por la moto, solo dan indicios de esa tendencia.